In [ ]:
from matplotlib.pylab import diff

from utils import carica_storico_json
from plot_utils import plot_training_history

def get_global_limits(histories, initial_gdv=None):
    all_losses = []
    all_accs = []
    all_gdvs = []
    
    if initial_gdv is not None:
        all_gdvs.append(initial_gdv)

    for h in histories:
        all_losses.extend(h['train_loss'])
        all_losses.extend(h['val_loss'])
        all_accs.extend(h['val_acc'])
        if 'test_acc' in h: all_accs.append(h['test_acc'])
        if 'val_gdv' in h: all_gdvs.extend(h['val_gdv'])

    # Calcolo limiti con piccolo margine (5%)
    def get_lim(data, margin=0.05):
        low, high = min(data), max(data)
        diff = high - low
        if diff == 0:  # Metrica costante su tutte le epoche → margine fisso per evitare asse a range nullo
            return (low - 0.1, high + 0.1)
        return (low - diff * margin, high + diff * margin)

    return {
        'loss': get_lim(all_losses),
        'acc': get_lim(all_accs),
        'gdv': get_lim(all_gdvs)
    }

# ==========================================
# 1. CONFIGURAZIONE DEL DATASET IN ANALISI
# Modifica queste variabili quando cambi dataset e log da analizzare!
# ==========================================
DATASET_ID = 59
NOME_DATASET = "Letter Recognition"
GDV_GREZZO = -0.1464

PATH_LOG_CE = f"results/{DATASET_ID}/logs/2026-05-04_18-30-00-holdout_cross_entropy.json"
PATH_LOG_POLY = f"results/{DATASET_ID}/logs/2026-05-04_18-35-00-holdout_polyloss.json"
PATH_LOG_CLOSS = f"results/{DATASET_ID}/logs/2026-05-04_18-40-00-holdout_closs.json"

# 2. CARICAMENTO DATI
history_ce = carica_storico_json(PATH_LOG_CE)
history_poly = carica_storico_json(PATH_LOG_POLY)
history_closs = carica_storico_json(PATH_LOG_CLOSS)

# 3. CALCOLO SCALA GLOBALE
# (Usa la stessa funzione get_global_limits definita sopra o incollala qui)
global_limits = get_global_limits([history_ce, history_poly, history_closs], initial_gdv=GDV_GREZZO)

# 4. GENERAZIONE GRAFICI UNIFICATI
print(f"=== Analisi Dataset: {NOME_DATASET} (ID: {DATASET_ID}) ===")

plot_training_history(history_ce, f"CE ({NOME_DATASET})", GDV_GREZZO, dataset_id=DATASET_ID, y_limits=global_limits)
plot_training_history(history_poly, f"Poly ({NOME_DATASET})", GDV_GREZZO, dataset_id=DATASET_ID, y_limits=global_limits)
plot_training_history(history_closs, f"C-Loss ({NOME_DATASET})", GDV_GREZZO, dataset_id=DATASET_ID, y_limits=global_limits)